# Fairness benchmark: base vs LoRA vs full fine-tuning on CrowS-Pairs

Runtime > Change runtime type > T4 GPU.

Requires the lora-fairness repo already cloned in this session (run
colab_lora_reproduction.ipynb's setup cells first, or the setup cell below
if starting fresh).

## 1. Setup (skip if already done in this session)

In [1]:
REPO_URL = "https://github.com/Hani0101/lora-fairness.git"
PROJECT_DIR = "/content/lora-fairness"

import os, shutil
from pathlib import Path

if REPO_URL:
    shutil.rmtree(PROJECT_DIR, ignore_errors=True)
    !git clone -q $REPO_URL $PROJECT_DIR
else:
    # Fallback: upload lora-fairness.zip when prompted.
    from google.colab import files
    uploaded = files.upload()
    archive = next(iter(uploaded))
    shutil.rmtree(PROJECT_DIR, ignore_errors=True)
    shutil.unpack_archive(archive, "/content")

os.chdir(PROJECT_DIR)
print(sorted(p.name for p in Path(".").iterdir()))

['.git', 'README.md', 'notebooks', 'requirements.txt', 'src', 'tests']


In [2]:
import os
os.chdir("/content/lora-fairness")
print(os.getcwd())

/content/lora-fairness


## 2. Train the CDA-augmented MLM checkpoints

Continued masked-language-model pretraining on a gender-augmented slice of
WikiText-2. Two runs: one LoRA, one full fine-tuning. This does not touch
CrowS-Pairs at all -- that stays held out for evaluation only.

In [3]:
!python -m src.fairness.train_mlm --method lora --model roberta-base --seed 0 --epochs 3

config.json: 100% 481/481 [00:00<00:00, 2.13MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 150kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 25.9MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 2.33MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 4.44MB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 26.7MB/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:  96% 701k/733k [00:00<00:00, 828kB/s]
wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes: 100% 732k/732k [00:00<00:00, 840kB/s, 72.1kB/s  ]
wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file: 100% 733k/733k [00:00<00:00, 841kB/s, 72.3kB/s  ]

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:  63% 3.99M/6.36M [00:00<00:00, 6.15MB/s]
wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes: 100% 6.35M/6.35M [00:00<00:00, 9.48MB/s,  626kB/s  ]
wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file: 100% 6.36M/6.36M [00:00<00:00, 9.49MB/s,  628kB/s  ]

wi

In [4]:
!python -m src.fairness.train_mlm --method full --model roberta-base --seed 0 --epochs 3

Map: 100% 7010/7010 [00:03<00:00, 1841.81 examples/s]
Loading weights: 100% 202/202 [00:00<00:00, 4162.41it/s]
{
  "total": 124697433,
  "trainable": 124697433,
  "lora_only": 0,
  "trainable_pct": 100.0
}
epoch 1 step 50/439 loss 1.8580
epoch 1 step 100/439 loss 1.7869
epoch 1 step 150/439 loss 1.7550
epoch 1 step 200/439 loss 1.7462
epoch 1 step 250/439 loss 1.7418
epoch 1 step 300/439 loss 1.7337
epoch 1 step 350/439 loss 1.7159
epoch 1 step 400/439 loss 1.7060
epoch 1 mean loss 1.6920
epoch 2 step 50/439 loss 1.5097
epoch 2 step 100/439 loss 1.5405
epoch 2 step 150/439 loss 1.5604
epoch 2 step 200/439 loss 1.5528
epoch 2 step 250/439 loss 1.5572
epoch 2 step 300/439 loss 1.5637
epoch 2 step 350/439 loss 1.5650
epoch 2 step 400/439 loss 1.5613
epoch 2 mean loss 1.5566
epoch 3 step 50/439 loss 1.5636
epoch 3 step 100/439 loss 1.5481
epoch 3 step 150/439 loss 1.5386
epoch 3 step 200/439 loss 1.5204
epoch 3 step 250/439 loss 1.5227
epoch 3 step 300/439 loss 1.5173
epoch 3 step 350/439 

## 3. Score all three: base, LoRA, full

Loads a fresh pretrained RoBERTa (untouched), then the LoRA checkpoint
injected on top of a fresh RoBERTa, then the full fine-tuned checkpoint --
and scores each on CrowS-Pairs (1,508 held-out sentence pairs, never used
in training).

In [5]:
LORA_CKPT = "runs_fairness/cda_roberta-base_lora_seed0/lora_weights.pt"
FULL_CKPT = "runs_fairness/cda_roberta-base_full_seed0/model.pt"

!python -m src.fairness.evaluate_bias \
    --model roberta-base \
    --lora-checkpoint {LORA_CKPT} \
    --full-checkpoint {FULL_CKPT}

Loaded 1508 CrowS-Pairs examples

=== base (no fine-tuning) ===
Loading weights: 100% 202/202 [00:00<00:00, 4815.31it/s]

=== lora ===
Loading weights: 100% 202/202 [00:00<00:00, 3688.70it/s]

=== full fine-tuning ===
Loading weights: 100% 202/202 [00:00<00:00, 4094.40it/s]

{
  "base": {
    "pct_stereotype": 59.35,
    "n_pairs": 1508,
    "n_more_preferred": 895,
    "by_bias_type": {
      "race-color": 54.07,
      "socioeconomic": 61.05,
      "gender": 54.96,
      "disability": 66.67,
      "nationality": 64.15,
      "sexual-orientation": 60.71,
      "physical-appearance": 60.32,
      "religion": 74.29,
      "age": 66.67
    }
  },
  "lora": {
    "pct_stereotype": 60.61,
    "n_pairs": 1508,
    "n_more_preferred": 914,
    "by_bias_type": {
      "race-color": 54.65,
      "socioeconomic": 61.05,
      "gender": 56.11,
      "disability": 66.67,
      "nationality": 65.41,
      "sexual-orientation": 69.05,
      "physical-appearance": 66.67,
      "religion": 75.24,
    

## 4. View the comparison

In [6]:
import json
import pandas as pd

results = json.loads(open("runs_fairness/bias_comparison.json").read())

rows = []
for method, r in results.items():
    row = {"method": method, "pct_stereotype": r["pct_stereotype"], "n_pairs": r["n_pairs"]}
    row["gender_pct"] = r["by_bias_type"].get("gender")
    rows.append(row)

df = pd.DataFrame(rows)
display(df)

,method,pct_stereotype,n_pairs,gender_pct
0,base,59.35,1508,54.96
1,lora,60.61,1508,56.11
2,full,58.95,1508,55.73


## Reading the numbers

`pct_stereotype` is the percentage of CrowS-Pairs sentence pairs where the
model assigned higher likelihood to the more-stereotypical sentence. 50% is
the unbiased baseline; higher means the model favors stereotypes more often.

`gender_pct` is the same metric restricted to CrowS-Pairs' gender category
only -- the one category the CDA fine-tuning actually targets. The overall
`pct_stereotype` also includes race, religion, age, and five other
categories the gender-swap augmentation does not address, so a shift in the
overall number that's smaller than the gender-only shift is expected, not a
bug.